# Lab 2 - Modelos Tradicionais

Classificadores: **KNN**, **Árvore de Decisão** e **SVM**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import os
from datetime import datetime

## Carregamento dos Dados Processados

In [ ]:
data_dir = './dados_processados'

X_train = pd.read_parquet(f'{data_dir}/X_train.parquet')
y_train = pd.read_parquet(f'{data_dir}/y_train.parquet').values.ravel()
X_test = pd.read_parquet(f'{data_dir}/X_test.parquet')
y_test = pd.read_parquet(f'{data_dir}/y_test.parquet').values.ravel()

mod_time = datetime.fromtimestamp(os.path.getmtime(f'{data_dir}/X_train.parquet'))
print(f'Dados carregados (gerados em {mod_time.strftime("%Y-%m-%d %H:%M")})')
print(f'  X_train: {X_train.shape}  |  X_test: {X_test.shape}')
print(f'  y_train: {y_train.shape}  |  y_test: {y_test.shape}')
print(f'  Features: {list(X_train.columns)}')
print(f'  Churn rate treino: {y_train.mean():.3f}  |  teste: {y_test.mean():.3f}')

## Função de Avaliação

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, cohen_kappa_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)


def avaliar_modelo(nome, y_true, y_pred):
    """Avalia um modelo e retorna dicionário com métricas."""
    metricas = {
        'Modelo': nome,
        'Acuracia': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F1-Score': f1_score(y_true, y_pred),
        'Kappa': cohen_kappa_score(y_true, y_pred)
    }
    print(f'\n=== {nome} ===')
    print(classification_report(y_true, y_pred, target_names=['Não cancelou', 'Cancelou']))
    print(f'Kappa: {metricas["Kappa"]:.4f}')
    return metricas


def salvar_metricas(resultados, arquivo):
    """Salva lista de dicts de métricas em JSON."""
    with open(f'{data_dir}/{arquivo}', 'w') as f:
        json.dump(resultados, f, indent=2)
    print(f'Métricas salvas em {data_dir}/{arquivo}')

## Normalização

KNN e SVM são sensíveis à escala das features — aplicamos `StandardScaler`.
Árvore de Decisão **não precisa** de normalização e usa `X_train`/`X_test` diretamente.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Dados normalizados — média ~0, std ~1')
print(f'  X_train_scaled: {X_train_scaled.shape}')
print(f'  X_test_scaled:  {X_test_scaled.shape}')